In [ ]:
!pip install langchain
!pip install openai
!pip install PyPDF2
!pip install faiss-cpu
!pip install tiktoken

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.0/90.0 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 101.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 22.0 MB/s eta 0:00:00


In [ ]:
from PyPDF2 import PdfReader
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import ElasticVectorSearch, Pinecone, Weaviate, FAISS

In [ ]:
# Get your API keys from openai, you will need to create an account.
# Here is the link to get the keys: https://platform.openai.com/account/billing/overview
import os
import getpass

if "OPENAI_API_KEY" not in os.environ or os.environ["OPENAI_API_KEY"] in ["", "XXXXXXXXXXXX"]:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key: ")

In [ ]:
# connect your Google Drive
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)
root_dir = "/content/gdrive/My Drive/"

Mounted at /content/gdrive


In [ ]:
# location of the pdf file/files.
reader = PdfReader('/content/gdrive/My Drive/coMAD.pdf')

In [ ]:
reader

In [ ]:
# read data from the file and put them into a variable called raw_text
raw_text = ''
for i, page in enumerate(reader.pages):
    text = page.extract_text()
    if text:
        raw_text += text

In [ ]:
# We need to split the text that we read into smaller chunks so that during information retreival we don't hit the token size limits.

text_splitter = CharacterTextSplitter(
    separator = "\n",
    chunk_size = 1000,
    chunk_overlap  = 200,
    length_function = len,
)
texts = text_splitter.split_text(raw_text)

In [ ]:
# Download embeddings from OpenAI
embeddings = OpenAIEmbeddings()

In [ ]:
docsearch = FAISS.from_texts(texts, embeddings)

IndexError: ignored

In [ ]:
docsearch

In [ ]:
from langchain.chains.question_answering import load_qa_chain
from langchain.llms import OpenAI

In [ ]:
chain = load_qa_chain(OpenAI(), chain_type="stuff")

In [ ]:
query = "What is a keyword tree ? Explain definition 5.3 of keyword tree explaining all the notations and symbols"
docs = docsearch.similarity_search(query)
chain.run(input_documents=docs, question=query)

" A keyword tree is a data structure used to store strings. It is a type of tree where each node is labeled with a letter and each edge is labeled with the letter that is used to traverse from one node to the next. The root node is labeled with a special symbol (often '$'), and each node is labeled with the prefix of the string that it represents. The leaves of the tree are labeled with the string that they represent. Each node also has an associated node-label, which is a list of strings stored at the node. The node-label of the root node is initialized to the list of all strings in the input set. As strings are added to the tree, they are removed from the node-label of the node they are being added to, and added to the node-labels of the nodes along the path. The node depth of a node uin a tree is the number of nodes on the path from the root to u."

In [ ]:
query = "What is Lemma 5.7 of the dictionary problem ? Expand the question and provide the answer in latex format"
docs = docsearch.similarity_search(query)
chain.run(input_documents=docs, question=query)

' Lemma 5.7 of the dictionary problem states that the dictionary problem can be solved in time O(jwj) with a preprocessing of time O(n). The statement can be expressed in latex as $\\text{Lemma } 5.7: \\text{The dictionary problem can be solved in time } O(jwj) \\text{ with a preprocessing of time } O(n)$.'